In [30]:
import time
import numpy as np
import pandas as pd

# Scikit-learn modules for Part A comparison
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [31]:
df = pd.read_csv("garments_worker_productivity.csv")

print("Shape:", df.shape)
df.head()

Shape: (1197, 15)


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


In [32]:
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())

print("\nColumns:")
print(df.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   str    
 1   quarter                1197 non-null   str    
 2   department             1197 non-null   str    
 3   day                    1197 non-null   str    
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float64
dtypes: float64(6), 

In [33]:
# Clean categorical strings (e.g., trailing spaces in 'finishing ')
df['department'] = df['department'].str.strip()

# Impute missing values in Work-In-Progress (wip) with 0
df['wip'] = df['wip'].fillna(0)

# Drop date column as it's an identifier
df = df.drop(columns=['date'])

# Create binary classification target
df['MeetsTarget'] = (df['actual_productivity'] >= df['targeted_productivity']).astype(int)

# One-hot encode categorical variables
cat_cols = ['quarter', 'department', 'day', 'team']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=float)

# Separate features and targets
# Note: Classification EXCLUDES actual_productivity as required
X_reg = df_encoded.drop(columns=['actual_productivity', 'MeetsTarget'])
y_reg = df_encoded['actual_productivity'].values.astype(np.float64)

X_cls = df_encoded.drop(columns=['actual_productivity', 'MeetsTarget'])
y_cls = df_encoded['MeetsTarget'].values.astype(np.float64)

print("Features shape:", X_reg.shape)
print("Class distribution:\n", pd.Series(y_cls).value_counts())

Features shape: (1197, 30)
Class distribution:
 1.0    875
0.0    322
Name: count, dtype: int64


In [34]:
# Create single fixed train-test split (80/20)
indices = np.arange(len(df_encoded))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

# Regression datasets
X_train_reg_raw = X_reg.iloc[train_idx].values.astype(np.float64)
X_test_reg_raw = X_reg.iloc[test_idx].values.astype(np.float64)
y_train_reg = y_reg[train_idx]
y_test_reg = y_reg[test_idx]

# Classification datasets
X_train_cls_raw = X_cls.iloc[train_idx].values.astype(np.float64)
X_test_cls_raw = X_cls.iloc[test_idx].values.astype(np.float64)
y_train_cls = y_cls[train_idx]
y_test_cls = y_cls[test_idx]

print(f"Train size: {len(train_idx)}, Test size: {len(test_idx)}")

Train size: 957, Test size: 240


In [35]:
# 1. Feature Scaling
scaler_sk_reg = StandardScaler()
X_train_reg_sk = scaler_sk_reg.fit_transform(X_train_reg_raw)
X_test_reg_sk = scaler_sk_reg.transform(X_test_reg_raw)

scaler_sk_cls = StandardScaler()
X_train_cls_sk = scaler_sk_cls.fit_transform(X_train_cls_raw)
X_test_cls_sk = scaler_sk_cls.transform(X_test_cls_raw)

# 2. Linear Regression Training & Inference
t0 = time.perf_counter()
sk_lr = LinearRegression()
sk_lr.fit(X_train_reg_sk, y_train_reg)
sk_lr_train_time = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
y_pred_lr_sk = sk_lr.predict(X_test_reg_sk)
sk_lr_pred_time = (time.perf_counter() - t0) * 1000

# Evaluate Linear Regression
sk_mae = mean_absolute_error(y_test_reg, y_pred_lr_sk)
sk_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_lr_sk))
sk_r2 = r2_score(y_test_reg, y_pred_lr_sk)

# 3. Logistic Regression Training & Inference
t0 = time.perf_counter()
sk_log = LogisticRegression(max_iter=1000, random_state=42)
sk_log.fit(X_train_cls_sk, y_train_cls)
sk_log_train_time = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
y_pred_log_sk = sk_log.predict(X_test_cls_sk)
sk_log_pred_time = (time.perf_counter() - t0) * 1000

# Evaluate Logistic Regression
sk_acc = accuracy_score(y_test_cls, y_pred_log_sk)
sk_prec = precision_score(y_test_cls, y_pred_log_sk)
sk_rec = recall_score(y_test_cls, y_pred_log_sk)
sk_f1 = f1_score(y_test_cls, y_pred_log_sk)

print("--- Scikit-Learn Results ---")
print(f"Regression -> MAE: {sk_mae:.4f}, RMSE: {sk_rmse:.4f}, R2: {sk_r2:.4f}")
print(f"Classification -> Accuracy: {sk_acc:.4f}, Precision: {sk_prec:.4f}, Recall: {sk_rec:.4f}, F1: {sk_f1:.4f}")

--- Scikit-Learn Results ---
Regression -> MAE: 0.1106, RMSE: 0.1495, R2: 0.1584
Classification -> Accuracy: 0.7583, Precision: 0.7931, Recall: 0.9096, F1: 0.8474


In [36]:
# Manual Standard Scaler
class ScratchStandardScaler:
    def fit_transform(self, X):
        X = np.asarray(X, dtype=np.float64)
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        self.std[self.std == 0] = 1e-8  # Prevent division by zero
        return (X - self.mean) / self.std

    def transform(self, X):
        X = np.asarray(X, dtype=np.float64)
        return (X - self.mean) / self.std

# Manual Metrics
def scratch_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def scratch_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def scratch_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

def scratch_classification_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    
    acc = (tp + tn) / len(y_true)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    return acc, prec, rec, f1

print("Manual utilities initialized.")

Manual utilities initialized.


In [37]:
# Closed-Form Linear Regression
class ScratchLinearRegression:
    def __init__(self):
        self.weights = None

    def fit(self, X, y):
        # Vectorized Normal Equation: w = (X^T X)^(-1) X^T y with bias column
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        self.weights = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y

    def predict(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b @ self.weights

# Gradient Descent Logistic Regression with L2 Regularization
class ScratchLogisticRegression:
    def __init__(self, lr=0.1, epochs=2000, reg=0.1):
        self.lr = lr
        self.epochs = epochs
        self.reg = reg
        self.w = None
        self.b = None

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -250, 250)))

    def fit(self, X, y):
        m, n = X.shape
        self.w = np.zeros(n)
        self.b = 0.0

        for _ in range(self.epochs):
            z = np.dot(X, self.w) + self.b
            y_hat = self._sigmoid(z)

            # Vectorized Gradients
            dw = (1 / m) * np.dot(X.T, (y_hat - y)) + (self.reg / m) * self.w
            db = (1 / m) * np.sum(y_hat - y)

            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict_proba(self, X):
        z = np.dot(X, self.w) + self.b
        return self._sigmoid(z)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

print("From-scratch models initialized.")

From-scratch models initialized.


In [38]:
# Scale Data Manually
scaler_m_reg = ScratchStandardScaler()
X_tr_reg_m = scaler_m_reg.fit_transform(X_train_reg_raw)
X_te_reg_m = scaler_m_reg.transform(X_test_reg_raw)

scaler_m_cls = ScratchStandardScaler()
X_tr_cls_m = scaler_m_cls.fit_transform(X_train_cls_raw)
X_te_cls_m = scaler_m_cls.transform(X_test_cls_raw)

# 1. Scratch Linear Regression Execution
t0 = time.perf_counter()
scratch_lr = ScratchLinearRegression()
scratch_lr.fit(X_tr_reg_m, y_train_reg)
scratch_lr_train_time = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
y_pred_lr_m = scratch_lr.predict(X_te_reg_m)
scratch_lr_pred_time = (time.perf_counter() - t0) * 1000

s_mae = scratch_mae(y_test_reg, y_pred_lr_m)
s_rmse = scratch_rmse(y_test_reg, y_pred_lr_m)
s_r2 = scratch_r2(y_test_reg, y_pred_lr_m)

# 2. Scratch Logistic Regression Execution
t0 = time.perf_counter()
scratch_log = ScratchLogisticRegression(lr=0.1, epochs=2000, reg=0.1)
scratch_log.fit(X_tr_cls_m, y_train_cls)
scratch_log_train_time = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
y_pred_log_m = scratch_log.predict(X_te_cls_m)
scratch_log_pred_time = (time.perf_counter() - t0) * 1000

s_acc, s_prec, s_rec, s_f1 = scratch_classification_metrics(y_test_cls, y_pred_log_m)

print("--- From-Scratch Results ---")
print(f"Regression -> MAE: {s_mae:.4f}, RMSE: {s_rmse:.4f}, R2: {s_r2:.4f}")
print(f"Classification -> Accuracy: {s_acc:.4f}, Precision: {s_prec:.4f}, Recall: {s_rec:.4f}, F1: {s_f1:.4f}")

--- From-Scratch Results ---
Regression -> MAE: 0.1106, RMSE: 0.1495, R2: 0.1584
Classification -> Accuracy: 0.7583, Precision: 0.7960, Recall: 0.9040, F1: 0.8466


In [39]:
reg_comparison = pd.DataFrame({
    'Metric / Property': ['MAE', 'RMSE', 'R2 Score', 'Train Time (ms)', 'Pred Time (ms)'],
    'Scikit-Learn': [f"{sk_mae:.4f}", f"{sk_rmse:.4f}", f"{sk_r2:.4f}", f"{sk_lr_train_time:.3f}", f"{sk_lr_pred_time:.3f}"],
    'From Scratch': [f"{s_mae:.4f}", f"{s_rmse:.4f}", f"{s_r2:.4f}", f"{scratch_lr_train_time:.3f}", f"{scratch_lr_pred_time:.3f}"]
})

cls_comparison = pd.DataFrame({
    'Metric / Property': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Train Time (ms)', 'Pred Time (ms)'],
    'Scikit-Learn': [f"{sk_acc:.4f}", f"{sk_prec:.4f}", f"{sk_rec:.4f}", f"{sk_f1:.4f}", f"{sk_log_train_time:.3f}", f"{sk_log_pred_time:.3f}"],
    'From Scratch': [f"{s_acc:.4f}", f"{s_prec:.4f}", f"{s_rec:.4f}", f"{s_f1:.4f}", f"{scratch_log_train_time:.3f}", f"{scratch_log_pred_time:.3f}"]
})

print("=== LINEAR REGRESSION COMPARISON ===")
print(reg_comparison.to_string(index=False))

print("\n=== LOGISTIC REGRESSION COMPARISON ===")
print(cls_comparison.to_string(index=False))

=== LINEAR REGRESSION COMPARISON ===
Metric / Property Scikit-Learn From Scratch
              MAE       0.1106       0.1106
             RMSE       0.1495       0.1495
         R2 Score       0.1584       0.1584
  Train Time (ms)        4.059        1.111
   Pred Time (ms)        0.466        0.219

=== LOGISTIC REGRESSION COMPARISON ===
Metric / Property Scikit-Learn From Scratch
         Accuracy       0.7583       0.7583
        Precision       0.7931       0.7960
           Recall       0.9096       0.9040
         F1-Score       0.8474       0.8466
  Train Time (ms)       13.475       64.611
   Pred Time (ms)        0.398        0.073


In [40]:
# Optimized Scratch Logistic Regression using Vectorized Adam Optimizer
class ScratchAdamLogisticRegression:
    def __init__(self, lr=0.01, epochs=1000, reg=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.epochs = epochs
        self.reg = reg
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -250, 250)))

    def fit(self, X, y):
        m, n = X.shape
        self.w = np.zeros(n)
        self.b = 0.0

        mw, vw = np.zeros(n), np.zeros(n)
        mb, vb = 0.0, 0.0

        for t in range(1, self.epochs + 1):
            z = np.dot(X, self.w) + self.b
            y_hat = self._sigmoid(z)

            dw = (1 / m) * np.dot(X.T, (y_hat - y)) + (self.reg / m) * self.w
            db = (1 / m) * np.sum(y_hat - y)

            # Adam Moment Updates
            mw = self.beta1 * mw + (1 - self.beta1) * dw
            vw = self.beta2 * vw + (1 - self.beta2) * (dw ** 2)
            mb = self.beta1 * mb + (1 - self.beta1) * db
            vb = self.beta2 * vb + (1 - self.beta2) * (db ** 2)

            mw_hat = mw / (1 - self.beta1 ** t)
            vw_hat = vw / (1 - self.beta2 ** t)
            mb_hat = mb / (1 - self.beta1 ** t)
            vb_hat = vb / (1 - self.beta2 ** t)

            self.w -= self.lr * mw_hat / (np.sqrt(vw_hat) + self.eps)
            self.b -= self.lr * mb_hat / (np.sqrt(vb_hat) + self.eps)

    def predict(self, X, threshold=0.5):
        z = np.dot(X, self.w) + self.b
        return (self._sigmoid(z) >= threshold).astype(int)

# Fit Optimized Model
t0 = time.perf_counter()
opt_log = ScratchAdamLogisticRegression(lr=0.05, epochs=800, reg=0.01)
opt_log.fit(X_tr_cls_m, y_train_cls)
t_opt_train = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
y_pred_opt = opt_log.predict(X_te_cls_m)
t_opt_pred = (time.perf_counter() - t0) * 1000

o_acc, o_prec, o_rec, o_f1 = scratch_classification_metrics(y_test_cls, y_pred_opt)

print(f"Optimized Scratch Logistic Regression (Adam) Accuracy: {o_acc:.4f}, F1: {o_f1:.4f}")
print(f"Train Time: {t_opt_train:.3f} ms, Prediction Time: {t_opt_pred:.3f} ms")

Optimized Scratch Logistic Regression (Adam) Accuracy: 0.7542, F1: 0.8443
Train Time: 37.198 ms, Prediction Time: 0.074 ms
